In [1]:
# CELL 1 — SIMPLE & CLEAN

!pip install -q --upgrade \
    "transformers==4.56.2" \
    "datasets==4.3.0" \
    "trl==0.24.0" \
    "accelerate" \
    "peft" \
    "bitsandbytes" \
    "sentencepiece" \
    "protobuf==5.29.6" \
    "pillow==11.3.0"

!pip install -q --upgrade \
    "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"

!pip install -q --upgrade \
    "unsloth @ git+https://github.com/unslothai/unsloth.git"

!pip install -q hf_transfer msgspec tyro cut_cross_entropy jedi

# TEST
import torch
import unsloth
import transformers
import datasets
import trl
import PIL
import torchvision

from unsloth import FastLanguageModel

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Pillow:", PIL.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("trl:", trl.__version__)

print("GPU:", torch.cuda.get_device_name(0))
print("Unsloth import OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 129.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 180.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 27.8 MB/s

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — CONFIG + CHECK DATA FILES  (v2 / 15k dataset)
# ═══════════════════════════════════════════════════════════

from pathlib import Path

DATA_DIR = Path("/content")

TRAIN_FILE = DATA_DIR / "acne_chat_response.train.jsonl"
VAL_FILE   = DATA_DIR / "acne_chat_response.val.jsonl"
TEST_FILE  = DATA_DIR / "acne_chat_response.test.jsonl"

OUTPUT_DIR = Path("/content/qwen25_acne_lora_v2")
MERGED_DIR = Path("/content/qwen25_acne_merged_v2")
GGUF_DIR   = Path("/content/qwen25_acne_gguf_v2")

BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 8
GRAD_ACCUM = 1
EPOCHS = 3
LR = 2e-4

print("TRAIN:", TRAIN_FILE, TRAIN_FILE.exists())
print("VAL:", VAL_FILE, VAL_FILE.exists())
print("TEST:", TEST_FILE, TEST_FILE.exists())

print("BASE_MODEL:", BASE_MODEL)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("GRAD_ACCUM:", GRAD_ACCUM)
print("EPOCHS:", EPOCHS)
print("LR:", LR)


In [3]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — LOAD MODEL + TOKENIZER
# ═══════════════════════════════════════════════════════════

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

print("Model loaded:", BASE_MODEL)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("pad_token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token, tokenizer.eos_token_id)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: unsloth/Qwen2.5-7B-Instruct-bnb-4bit
Tokenizer class: Qwen2TokenizerFast
pad_token: <|PAD_TOKEN|> 151665
eos_token: <|im_end|> 151645
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM GB: 94.97


In [8]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — LOAD DATASET + PREVIEW STRUCTURE
# ═══════════════════════════════════════════════════════════

from datasets import load_dataset
import json

data_files = {
    "train": str(TRAIN_FILE),
    "validation": str(VAL_FILE),
    "test": str(TEST_FILE),
}

dataset = load_dataset("json", data_files=data_files)

print(dataset)

sample = dataset["train"][0]

print("\n===== KEYS =====")
print(sample.keys())

print("\n===== METADATA =====")
print(json.dumps(sample.get("metadata", {}), ensure_ascii=False, indent=2)[:2000])

print("\n===== MESSAGES =====")
for i, msg in enumerate(sample["messages"]):
    print(f"\n--- message {i} ---")
    print("role:", msg.get("role"))
    print("content preview:")
    print(str(msg.get("content", ""))[:1500])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 54233
    })
    validation: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 3247
    })
    test: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 3169
    })
})

===== KEYS =====
dict_keys(['messages', 'metadata'])

===== METADATA =====
{
  "id": "1f3ae870a33d934277fb0e47",
  "sample_type": "clinical_chat_response",
  "source_chunk_id": "vinmec_7c6f596a16::0000",
  "source_title": "Mụn trứng cá trong kỳ kinh nguyệt: Nguyên nhân và cách điều trị",
  "source_url": "https://www.vinmec.com/vie/bai-viet/mun-trung-ca-trong-ky-kinh-nguyet-nguyen-nhan-va-cach-dieu-tri-vi",
  "source_trust_level": "high",
  "case_type": "pregnancy_breastfeeding_safety",
  "severity": "moderate",
  "safety_flags": [
    "pregnancy"
  ],
  "guideline_refs": [
    "AAD_2024",
    "NICE_NG198"
  ],
  "confidence": "high",
  "validator_version": "v1",
  "rule_filter": {
  

In [9]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — FORMAT MESSAGES TO CHAT TEMPLATE TEXT
# ═══════════════════════════════════════════════════════════

def format_messages_to_text(example):
    messages = example["messages"]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}

dataset = dataset.map(
    format_messages_to_text,
    remove_columns=["messages", "metadata"],
    desc="Formatting ChatML text",
)

print(dataset)

print("\n===== SAMPLE TEXT =====")
print(dataset["train"][0]["text"][:2500])

Formatting ChatML text:   0%|          | 0/54233 [00:00<?, ? examples/s]

Formatting ChatML text:   0%|          | 0/3247 [00:00<?, ? examples/s]

Formatting ChatML text:   0%|          | 0/3169 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 54233
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3247
    })
    test: Dataset({
        features: ['text'],
        num_rows: 3169
    })
})

===== SAMPLE TEXT =====
<|im_start|>system
Bạn là AcneLLM, trợ lý tư vấn da mụn tiếng Việt. Trả lời ngắn gọn, rõ ràng, an toàn y tế là ưu tiên cao nhất. Không chẩn đoán chắc chắn, không kê đơn thuốc, không đưa liều thuốc kê đơn (isotretinoin, kháng sinh uống, retinoid kê đơn, corticoid). Khi mô tả gợi ý mụn nặng, mụn bọc/nang, sẹo, lan rộng, nhiễm trùng, hoặc đang mang thai/cho con bú, hãy khuyên đi khám bác sĩ da liễu và tránh tự dùng thuốc. Nếu thiếu dữ kiện thì hỏi thêm thay vì đoán. Bám AAD 2024 + NICE NG198, không bịa.<|im_end|>
<|im_start|>user
<raw_user_message>Em 30 tuổi, có thai 3 tháng. Gần đây mặt em nổi nhiều mụn đỏ, có mủ, tập trung ở má và cằm. Em lo lắng vì không dám dùng thuốc. Em nên làm gì ạ?</raw_user_message>
<

In [10]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — TOKEN LENGTH CHECK
# ═══════════════════════════════════════════════════════════

def count_tokens(example):
    ids = tokenizer(
        example["text"],
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    return {"token_len": len(ids)}

dataset_len = dataset.map(
    count_tokens,
    desc="Counting tokens",
)

train_lens = dataset_len["train"]["token_len"]
val_lens = dataset_len["validation"]["token_len"]

def show_stats(name, arr):
    arr_sorted = sorted(arr)
    n = len(arr_sorted)

    def pct(p):
        idx = int(n * p / 100)
        idx = min(idx, n - 1)
        return arr_sorted[idx]

    print(f"\n===== {name} =====")
    print("samples:", n)
    print("min:", arr_sorted[0])
    print("avg:", round(sum(arr_sorted) / n, 2))
    print("p50:", pct(50))
    print("p90:", pct(90))
    print("p95:", pct(95))
    print("p99:", pct(99))
    print("max:", arr_sorted[-1])
    print(f"> MAX_SEQ_LENGTH {MAX_SEQ_LENGTH}:", sum(x > MAX_SEQ_LENGTH for x in arr_sorted))

show_stats("TRAIN", train_lens)
show_stats("VALIDATION", val_lens)

Counting tokens:   0%|          | 0/54233 [00:00<?, ? examples/s]

Counting tokens:   0%|          | 0/3247 [00:00<?, ? examples/s]

Counting tokens:   0%|          | 0/3169 [00:00<?, ? examples/s]


===== TRAIN =====
samples: 54233
min: 334
avg: 613.21
p50: 626
p90: 700
p95: 723
p99: 772
max: 943
> MAX_SEQ_LENGTH 1024: 0

===== VALIDATION =====
samples: 3247
min: 357
avg: 595.17
p50: 619
p90: 691
p95: 716
p99: 761
max: 854
> MAX_SEQ_LENGTH 1024: 0


In [11]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — ADD LORA ADAPTER
# ═══════════════════════════════════════════════════════════

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    # Unsloth checkpointing giúp giảm VRAM
    use_gradient_checkpointing = "unsloth",

    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("LoRA attached OK")

trainable = 0
total = 0

for _, param in model.named_parameters():
    total += param.numel()
    if param.requires_grad:
        trainable += param.numel()

print("Trainable params:", f"{trainable:,}")
print("Total params:", f"{total:,}")
print("Trainable %:", round(trainable / total * 100, 4))

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA attached OK
Trainable params: 40,370,176
Total params: 4,393,342,464
Trainable %: 0.9189


In [12]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — CREATE TRAINER
# ═══════════════════════════════════════════════════════════

from trl import SFTTrainer
from transformers import TrainingArguments
import torch

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir = str(OUTPUT_DIR),

    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = 1,
    gradient_accumulation_steps = GRAD_ACCUM,

    num_train_epochs = EPOCHS,
    learning_rate = LR,
    warmup_ratio = 0.03,

    logging_steps = 10,
    eval_steps = 250,
    save_steps = 250,
    save_total_limit = 2,

    fp16 = not bf16_ok,
    bf16 = bf16_ok,

    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",

    report_to = "none",
    remove_unused_columns = False,

    # Không dùng group_by_length để tránh lỗi version
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,

    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],

    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    packing = False,

    args = training_args,
)

print("Trainer created OK")
print("bf16:", bf16_ok)
print("train samples:", len(dataset["train"]))
print("val samples:", len(dataset["validation"]))

steps_per_epoch = len(dataset["train"]) // (BATCH_SIZE * GRAD_ACCUM)
print("approx steps per epoch:", steps_per_epoch)

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/54233 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/3247 [00:00<?, ? examples/s]

Trainer created OK
bf16: True
train samples: 54233
val samples: 3247
approx steps per epoch: 6779


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — REAL TRAINING OVERNIGHT / H100  (v2)
# ═══════════════════════════════════════════════════════════

import torch
from pathlib import Path

# v2 dataset run
OUTPUT_DIR = Path("/content/qwen25_acne_lora_v2_h100_b4_ga2")

trainer.args.output_dir = str(OUTPUT_DIR)

# Train thật
trainer.args.max_steps = -1
trainer.args.num_train_epochs = EPOCHS

# Log vừa đủ
trainer.args.logging_steps = 20

# Eval/save mỗi 500 steps (~4-5 lần/epoch với 13k samples)
trainer.args.eval_strategy = "steps"
trainer.args.eval_steps = 500

trainer.args.save_strategy = "steps"
trainer.args.save_steps = 500
trainer.args.save_total_limit = 3

print("Output dir:", trainer.args.output_dir)
print("Epochs:", trainer.args.num_train_epochs)
print("Max steps:", trainer.args.max_steps)
print("Save every:", trainer.args.save_steps)
print("Eval every:", trainer.args.eval_steps)
print("Batch size:", trainer.args.per_device_train_batch_size)
print("Grad accum:", trainer.args.gradient_accumulation_steps)
print("Effective batch:", trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps)
print("BF16:", trainer.args.bf16)
print("GPU:", torch.cuda.get_device_name(0))

train_result = trainer.train(resume_from_checkpoint=False)

print("Training finished")
print(train_result)

In [14]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — SAVE LORA ADAPTER
# ═══════════════════════════════════════════════════════════

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print("Saved LoRA adapter to:", OUTPUT_DIR)

!find {OUTPUT_DIR} -maxdepth 2 -type f | head -50

Saved LoRA adapter to: /content/qwen25_acne_lora_h100_b4_ga2
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/rng_state.pth
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/training_args.bin
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/trainer_state.json
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/scheduler.pt
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/merges.txt
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/optimizer.pt
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/vocab.json
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/added_tokens.json
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/tokenizer.json
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/adapter_model.safetensors
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/tokenizer_config.json
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/special_tokens_map.json
/content/qwen25_acne_lora_h100_b4_ga2/checkpoint-5000/chat_template.jinja
/content/qwen

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — TEST INFERENCE
# ═══════════════════════════════════════════════════════════

from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)

# Updated system prompt — natural-tone v2 (matches writer.py SYSTEM_CONTENT)
SYSTEM_CONTENT = (
    "Bạn là AcneLLM, trợ lý tư vấn da mụn tiếng Việt. "
    "Trả lời như một người bạn có kiến thức da liễu — tự nhiên, gần gũi, "
    "đi thẳng vào vấn đề ngay câu đầu tiên mà không chào hỏi hay dẫn dắt. "
    "Tone linh hoạt theo mức độ: mụn nhẹ/routine thì ngắn gọn thoải mái; "
    "tình huống phức tạp thì giải thích rõ hơn nhưng vẫn dễ đọc. "
    "Lưu ý an toàn chỉ thêm khi thực sự cần — mụn nặng, nodule/cyst, sẹo, "
    "nhiễm trùng, thai kỳ — thì khuyên gặp bác sĩ da liễu rõ ràng. "
    "Không chẩn đoán xác định, không kê đơn, không đưa liều thuốc kê đơn. "
    "Thiếu thông tin thì hỏi thêm thay vì đoán. Bám AAD 2024 + NICE NG198."
)

def acne_ask(raw_user_message, guideline_context="", max_new_tokens=512):
    user_content = f"""<raw_user_message>{raw_user_message}</raw_user_message>"""

    if guideline_context.strip():
        user_content += f"\n<guideline_context>{guideline_context}</guideline_context>"

    messages = [
        {
            "role": "system",
            "content": SYSTEM_CONTENT
        },
        {
            "role": "user",
            "content": user_content
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Cắt phần assistant cho dễ đọc
    if "<|im_start|>assistant" in decoded:
        decoded = decoded.split("<|im_start|>assistant")[-1]
    decoded = decoded.replace("<|im_end|>", "").strip()

    return decoded

print(acne_ask(
    "Em 21 tuổi, da dầu, mụn đỏ có mủ ở má và cằm khoảng 2 tháng, có nên tự mua isotretinoin uống không?"
))

In [16]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — EVALUATE VALIDATION LOSS
# ═══════════════════════════════════════════════════════════

eval_result = trainer.evaluate()

print("Eval result:")
print(eval_result)

Eval result:
{'eval_loss': 0.24403463304042816, 'eval_runtime': 156.7375, 'eval_samples_per_second': 20.716, 'eval_steps_per_second': 20.716, 'epoch': 1.0}


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14 — ZIP LORA ADAPTER  (v2)
# ═══════════════════════════════════════════════════════════

import shutil
from pathlib import Path

lora_zip = "/content/qwen25_acne_lora_v2.zip"

if Path(lora_zip).exists():
    Path(lora_zip).unlink()

shutil.make_archive(
    lora_zip.replace(".zip", ""),
    "zip",
    str(OUTPUT_DIR)
)

print("Created:", lora_zip)

!ls -lh {lora_zip}

In [18]:
# ═══════════════════════════════════════════════════════════
# CELL 15 — MERGE LORA INTO 16BIT MODEL
# ═══════════════════════════════════════════════════════════

MERGED_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained_merged(
    str(MERGED_DIR),
    tokenizer,
    save_method="merged_16bit",
)

print("Merged 16-bit model saved to:", MERGED_DIR)

!find {MERGED_DIR} -maxdepth 2 -type f | head -50

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

... (6 outputs left out)

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:39<00:00,  9.82s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:02<00:00, 15.51s/it]


Unsloth: Merge process complete. Saved to `/content/qwen25_acne_merged`
Merged 16-bit model saved to: /content/qwen25_acne_merged
/content/qwen25_acne_merged/merges.txt
/content/qwen25_acne_merged/config.json
/content/qwen25_acne_merged/model-00001-of-00004.safetensors
/content/qwen25_acne_merged/model-00004-of-00004.safetensors
/content/qwen25_acne_merged/vocab.json
/content/qwen25_acne_merged/added_tokens.json
/content/qwen25_acne_merged/tokenizer.json
/content/qwen25_acne_merged/tokenizer_config.json
/content/qwen25_acne_merged/model.safetensors.index.json
/content/qwen25_acne_merged/special_tokens_map.json
/content/qwen25_acne_merged/model-00003-of-00004.safetensors
/content/qwen25_acne_merged/chat_template.jinja
/content/qwen25_acne_merged/model-00002-of-00004.safetensors


In [19]:
# ═══════════════════════════════════════════════════════════
# CELL 16 — EXPORT GGUF Q4_K_M FOR OLLAMA
# ═══════════════════════════════════════════════════════════

GGUF_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

print("GGUF exported to:", GGUF_DIR)

!find {GGUF_DIR} -maxdepth 2 -type f
!ls -lh {GGUF_DIR}

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  25%|██▌       | 1/4 [00:07<00:21,  7.23s/it]

... (6 outputs left out)

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:53<00:00, 13.41s/it]


Unsloth: Merge process complete. Saved to `/content/qwen25_acne_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/qwen25_acne_gguf_gguf/Qwen2.5-7B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/qwen25_acne_gguf_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/qwen25_acne_gguf_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/qwen25_acne_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/qwen25_acne_gguf_gguf/Modelfile
GGUF exported to: /content/qwen25_acne_gguf
/content/qwen25_acne_gguf/merges.txt
/content/qwen25_acne_gguf/config.json
/content/qwen25_acne_gguf/model-00001-of-00004.safetensors
/content/

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 16b — EXPORT GGUF Q2_K_M  (for comparison with Q4_K_M)
# ═══════════════════════════════════════════════════════════

# Export thêm Q2_K_M để so sánh chất lượng vs Q4_K_M
# Lưu vào folder riêng để dễ quản lý

GGUF_DIR_Q2 = Path("/content/qwen25_acne_gguf_v2_q2km")
GGUF_DIR_Q2.mkdir(parents=True, exist_ok=True)

model.save_pretrained_gguf(
    str(GGUF_DIR_Q2),
    tokenizer,
    quantization_method="q2_k",
)

print("GGUF Q2_K_M exported to:", GGUF_DIR_Q2)

!find {GGUF_DIR_Q2} -maxdepth 2 -type f
!ls -lh {GGUF_DIR_Q2}

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 17 — ZIP GGUF  (v2)
# ═══════════════════════════════════════════════════════════

import shutil
from pathlib import Path

gguf_zip = "/content/qwen25_acne_v2_gguf_q4_k_m.zip"

if Path(gguf_zip).exists():
    Path(gguf_zip).unlink()

shutil.make_archive(
    gguf_zip.replace(".zip", ""),
    "zip",
    str(GGUF_DIR)
)

print("Created:", gguf_zip)

!ls -lh {gguf_zip}

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 17b — ZIP GGUF Q2_K_M  (v2)
# ═══════════════════════════════════════════════════════════

import shutil
from pathlib import Path

gguf_q2_zip = "/content/qwen25_acne_v2_gguf_q2_k_m.zip"

if Path(gguf_q2_zip).exists():
    Path(gguf_q2_zip).unlink()

shutil.make_archive(
    gguf_q2_zip.replace(".zip", ""),
    "zip",
    str(GGUF_DIR_Q2)
)

print("Created:", gguf_q2_zip)

!ls -lh {gguf_q2_zip}

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 18 — DOWNLOAD RESULTS  (v2)
# ═══════════════════════════════════════════════════════════

from google.colab import files

# Tải LoRA adapter v2
files.download("/content/qwen25_acne_lora_v2.zip")

# Tải GGUF Q4_K_M
files.download("/content/qwen25_acne_v2_gguf_q4_k_m.zip")

# Tải GGUF Q2_K_M (để so sánh)
files.download("/content/qwen25_acne_v2_gguf_q2_k_m.zip")

In [21]:
# ═══════════════════════════════════════════════════════════
# CELL 19 — PRINT OLLAMA MODELFILE TEMPLATE
# ═══════════════════════════════════════════════════════════

modelfile = r'''
FROM ./unsloth.Q4_K_M.gguf

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.05
PARAMETER num_ctx 2048

SYSTEM """
Bạn là AcneLLM, trợ lý tư vấn da mụn tiếng Việt.

Yêu cầu:
- Trả lời ngắn gọn, rõ ràng, an toàn.
- Không chẩn đoán chắc chắn.
- Không kê đơn thuốc.
- Không đưa liều thuốc kê đơn như isotretinoin, kháng sinh uống, retinoid kê đơn, corticoid.
- Nếu người dùng có mụn nặng, mụn nang/cục, đau nhiều, sẹo, lan rộng, nhiễm trùng, đang mang thai hoặc cho con bú, hãy khuyên đi khám bác sĩ da liễu.
- Nếu thiếu dữ kiện thì hỏi thêm thay vì đoán.
"""
'''

print(modelfile)


FROM ./unsloth.Q4_K_M.gguf

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.05
PARAMETER num_ctx 2048

SYSTEM """
Bạn là AcneLLM, trợ lý tư vấn da mụn tiếng Việt.

Yêu cầu:
- Trả lời ngắn gọn, rõ ràng, an toàn.
- Không chẩn đoán chắc chắn.
- Không kê đơn thuốc.
- Không đưa liều thuốc kê đơn như isotretinoin, kháng sinh uống, retinoid kê đơn, corticoid.
- Nếu người dùng có mụn nặng, mụn nang/cục, đau nhiều, sẹo, lan rộng, nhiễm trùng, đang mang thai hoặc cho con bú, hãy khuyên đi khám bác sĩ da liễu.
- Nếu thiếu dữ kiện thì hỏi thêm thay vì đoán.
"""



In [ ]:
# ═══════════════════════════════════════════════════════════
# BACKUP TO GOOGLE DRIVE — FOLDER: Finetune_v2  (v2 dataset)
# ═══════════════════════════════════════════════════════════

from google.colab import drive
from pathlib import Path
from datetime import datetime
import shutil
import os

# 1) Mount Google Drive
drive.mount("/content/drive")

# 2) Tạo folder backup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

DRIVE_ROOT = Path("/content/drive/MyDrive")
BACKUP_ROOT = DRIVE_ROOT / "Finetune_v2"
BACKUP_DIR = BACKUP_ROOT / f"content_backup_{timestamp}"

BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("Backup folder:", BACKUP_DIR)

# 3) Backup toàn bộ /content nhưng tránh copy chính Google Drive gây lặp vô hạn
EXCLUDE_NAMES = {
    "drive",
    "sample_data",
    "__pycache__",
}

for item in Path("/content").iterdir():
    if item.name in EXCLUDE_NAMES:
        print("Skip:", item)
        continue

    dest = BACKUP_DIR / item.name

    try:
        if item.is_dir():
            print("Copy dir:", item, "->", dest)
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            print("Copy file:", item, "->", dest)
            shutil.copy2(item, dest)
    except Exception as e:
        print("FAILED:", item, "|", repr(e))

# 4) Tạo manifest liệt kê file đã backup
manifest_path = BACKUP_DIR / "BACKUP_MANIFEST.txt"

with open(manifest_path, "w", encoding="utf-8") as f:
    f.write(f"Backup time: {timestamp}\n")
    f.write(f"Source: /content\n")
    f.write(f"Destination: {BACKUP_DIR}\n\n")
    f.write("Files:\n")

    for path in BACKUP_DIR.rglob("*"):
        if path.is_file():
            try:
                size_mb = path.stat().st_size / 1024 / 1024
                f.write(f"{path.relative_to(BACKUP_DIR)} | {size_mb:.2f} MB\n")
            except:
                f.write(f"{path.relative_to(BACKUP_DIR)}\n")

# 5) Nhắc lưu notebook
readme_path = BACKUP_DIR / "NOTEBOOK_BACKUP_NOTE.txt"

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(
        "Colab notebook .ipynb không nằm trực tiếp trong /content.\n"
        "Để lưu notebook, vào Colab: File -> Save a copy in Drive.\n"
        "Folder backup này đã lưu toàn bộ file/thư mục trong /content, trừ /content/drive và sample_data.\n"
    )

print("\nDONE.")
print("Saved to:", BACKUP_DIR)
print("Manifest:", manifest_path)

!du -sh "{BACKUP_DIR}"
!find "{BACKUP_DIR}" -maxdepth 2 -type f | head -50

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 20 — LOAD MODEL INTO OLLAMA
# ═══════════════════════════════════════════════════════════

# Run these after copying the .gguf + Modelfile to your local machine
# or run directly in Colab if Ollama is installed in the runtime

!ollama create acnellm-qwen25-v2 -f /content/qwen25_acne_gguf_v2_gguf/Modelfile
!ollama run acnellm-qwen25-v2